In [1]:
from langchain_community.document_loaders import PyPDFLoader


from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama, OllamaEmbeddings

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableLambda, RunnableParallel, RunnablePassthrough

from dotenv import load_dotenv
load_dotenv()


# load PDF
file_path = "data/Student Handbook 2025_ENG.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load() 
transcript = " ".join([doc.page_content for doc in docs])

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks= splitter.create_documents([transcript])


# loading vector store

model = "Ollama"
embedding = OllamaEmbeddings(model="mxbai-embed-large")
try:
    vector_store = FAISS.load_local("student_handbook_db", embedding, index_name=model, allow_dangerous_deserialization=True)
except Exception:
    vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)
    vector_store.save_local("student_handbook_db", index_name=model)



llm = ChatOllama(model= "mistral")
parser = StrOutputParser()


retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={'k': 30, 'lambda_mult': 0.5})

aug_prompt = PromptTemplate(
                            template="""
                            You are a helpful assistant. 
                            Answer ONLY from the provided transcript context. 
                            Answer in bullet form.
                            If the context is insufficient, just say you don't know.
                            Transcript: {context}
                            Question: {question}
                            """, 
                            input_variables=["context", "question"]
                            )

# chains
context_chain = RunnableSequence(retriever, RunnableLambda(lambda retrieved_docs: "\n\n".join([doc.page_content for doc in retrieved_docs])))
aug_prompt_chain = RunnableSequence(aug_prompt, llm)


parallel_chain = RunnableParallel({"context": context_chain, "question": RunnablePassthrough()})

chain = RunnableSequence(parallel_chain, aug_prompt_chain, parser)


# with open("questions.txt", "r", encoding="utf-8") as f:
#     questions = [line.strip() for line in f if line.strip()]
    
# answers = chain.batch(questions, config={"max_concurrency": 4})

# results = dict(zip(questions, answers))
# with open(f"qa_results_{model}.txt", "w", encoding="utf-8") as f:
#     for q, a in results.items():
#         f.write(f"Q: {q}\n")
#         f.write(f"A: {a}\n")
#         f.write("-"*50 + "\n\n")


# question = "who is the author of this document?"
# question = "what is class hours?"
# question = "what is leave policy?"
# question = "explain seminars"
# question = "i do not understand the grp section"
# question = "what all courses in Data Science?"
# question = "which data science courses are offered in the Fall semester?"
# question = "which data science courses are offered in the Spring semester?"
# question = "fall semester finance courses?"
# question = "fall semester data science courses"
# question = "pre-requisite for Mathematics for Data Science"
# question = "pre-requisite for Machine Learning"
# question = "how many semesters a student need to spend to graduate?"

# question = "what is the japanese language requirement for graduation?"
# question = "how many workshops credits are necessary to graduate?"
# question = "major requirement for psychology?"
# question = "major requirement for data science?"
# question = "major requirement for political science?"
# question = "pre-requisite for Political Economy of Development"
# question = "I want to take a leave, what should I do?"
question = "what is the minimum credits required for graduation?"

result = chain.invoke(question)
print(result)

 - To graduate, students must earn a total of 124 credits or more from courses taught at iCLA.
- For students who entered in 2024AY or earlier:
  - At least 4 must be "designated courses" (12 credits) that satisfy the major area Graduation Requirements.
  - Psychology majors must have earned credit for at least 7 lecture courses (21 credits) that satisfy the major area requirement.
- For students who entered in 2025 or later:
  - English Language Courses: 'English for Academic Excellence: B (EAE B)' (15 credits) is required.
  - Major area courses: Must earn 6 lecture courses * (33 credits or 30 credits, depending on the major area chosen).
  - 'Seminar' (1 credit) and 'Graduation Research Project' (2 credits) are also required.


In [4]:
# question = "who is the dean?"

# question = "how much compensation is given in case of death on the campus?"


question = "I want to take a leave, what should I do?"
result = chain.invoke(question)
print(result)

 To take a leave, follow these steps:

- Determine the duration of your leave (1 or 2 semesters)
- Apply for Leave of Absence with permission from the University President
- If approved, international students must leave Japan. When re-entering Japan after the Leave of Absence, students must re-apply for a Certificate of Eligibility
- During a Leave of Absence, your year level and semester will remain the same, but it counts towards your Maximum Enrollment Period
